In [ ]:
!pip install -q transformers emoji emoticon_fix torch

In [ ]:
import torch, json, re, emoji
import pandas as pd
import numpy as np
from emoticon_fix import emoticon_fix
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

In [ ]:
df = pd.read_csv("Mental_Distress_Dataset_updated.csv")

In [ ]:
df_test = pd.read_csv("mental_distress_test_set.csv")

In [ ]:
def preprocess_text(text):
  text = text.lower()
  """
  '\s+'- matches miltiple spaces/tabs/newlines, replacing with one space only
  strip() helps to remove spaces at the beginning and end
  """
  text = re.sub(r'\s+', ' ', text).strip()
  text = emoji.demojize(text)
  text = emoticon_fix(text)
  text = re.sub(r'@\w+', '', text) #  # @\w+ matches things like @john

  return text

# Apply preprocessing
df['text'] = df['text'].apply(preprocess_text)

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_10801/2448438152.py:4: SyntaxWarning: invalid escape sequence '\s'
  '\s+'- matches miltiple spaces/tabs/newlines, replacing with one space only


In [ ]:
with open("label_mapping.json", "r") as f:
    label_mapping = json.load(f)

id_to_label = {v: k for k, v in label_mapping.items()}

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Encode test texts
test_encodings = tokenizer(
    list(df_test['text']),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
# Create Dataset class
class MentalDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item
    def __len__(self):
        return len(self.labels)

In [ ]:
test_dataset = MentalDataset(test_encodings, df_test['label_encoded'])
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
num_labels = len(label_mapping)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

state_dict = torch.load("distilbert_emotion_model.pth", map_location='cpu')
model.load_state_dict(state_dict)
model.eval()

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
preds, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

target_names = [id_to_label[i] for i in sorted(id_to_label.keys())]

print("✅ Classification report on original test set:")
print(classification_report(true_labels, preds, target_names=target_names, digits=4))

✅ Classification report on original test set:
              precision    recall  f1-score   support

     Anxious     0.9284    0.9534    0.9407       408
   Depressed     0.6903    0.6415    0.6650       410
  Frustrated     0.9270    0.8932    0.9098       412
      Others     0.8806    0.8287    0.8538       356
    Suicidal     0.7275    0.8180    0.7701       434

    accuracy                         0.8267      2020
   macro avg     0.8307    0.8269    0.8279      2020
weighted avg     0.8282    0.8267    0.8265      2020



In [ ]:
"""
Numeric labels to textual labels to stress scores
"""

# Converts emotion labels → numeric stress values
stress_mapping = {
    'Suicidal': 5,
    'Depressed': 4,
    'Anxious': 3,
    'Frustrated': 2,
    'Others': 1
}

# Convert model predictions (numeric) → stress scores
pred_labels = [id_to_label[i] for i in preds]  # preds from previous step
stress_scores = [stress_mapping[label] for label in pred_labels] # pred_labels - DistilBERT predictions

In [ ]:
batch_size = 16  # same as training
aggregated_scores = []

# main computation
def aggregate_stress_scores(scores, batch_size=16):
    return [np.mean(scores[i:i+batch_size]) for i in range(0, len(scores), batch_size)]

aggregated_scores = aggregate_stress_scores(stress_scores)
print(aggregated_scores[:5])

[np.float64(3.3125), np.float64(2.5625), np.float64(2.9375), np.float64(3.5), np.float64(3.5)]


In [ ]:
# Normalize to 0-10 scale (main function)

def normalize_score(score, min_score=1, max_score=5):
    return (score - min_score) / (max_score - min_score) * 10

In [ ]:
# 0–4 → Low, 4–7 → Medium, 7–10 → High

def get_stress_level(score):
    if score <= 4:
        return "Low"
    elif score <= 7:
        return "Medium"
    else:
        return "High"

In [ ]:
# Function to predict stress for a single user input
def predict_user_input(text):

    # Preprocess the input text
    processed_text = preprocess_text(text)

    # Tokenize
    encoding = tokenizer(
        [processed_text],
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors='pt'  # returns PyTorch tensors
    )

    # Move to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Model prediction
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        pred_id = torch.argmax(logits, dim=-1).item()  # single prediction

    # Map prediction to label
    pred_label = id_to_label[pred_id]

    stress_score = stress_mapping[pred_label]   # label → number
    normalized_score = normalize_score(stress_score)  # number → scaled
    stress_level = get_stress_level(normalized_score) # scaled → category

    return pred_label, stress_score, normalized_score, stress_level

In [ ]:
# List to store stress scores for all user inputs
user_stress_scores = []

num_posts = int(input("How many posts do you want to input? "))

"""
4 tasks inside the function:
  Take input
  Predict
  Print result
  Store scores
"""
for i in range(num_posts):
    user_text = input(f"Enter post {i+1}: ")

    label, score, norm_score, level = predict_user_input(user_text)

    print(f"Post {i+1}: {label} | Score: {score} | Normalized: {norm_score:.2f} | Level: {level}\n")

    user_stress_scores.append(score)

# Aggregate user inputs
batch_avg = np.mean(user_stress_scores)

# Normalize aggregated scores
normalized_batch_score = normalize_score(batch_avg)

# Final stress level
batch_stress_level = get_stress_level(normalized_batch_score)

print(f"\nAggregated Stress Score (normalized 0-10): {normalized_batch_score:.2f}")
print(f"Aggregated Stress Level for your posts: {batch_stress_level}")